# LiDAR Spoofing and Defense Practice

The assignment for this notebook is to use geometric operations usign Open3D to inject spoofed points into a LiDAR scan and learn trivial defense techniques.

# 1. Point Cloud Data Loading and Visualization

In [ ]:
# Install the Open3D library
# This step is needed because Open3D is not a standard library included in Google Colab
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.7/399.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.2 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.1.2
    Uninstalling Werkzeug-3.1.2:
      Successfully uninstalled Werkzeug-3.1.2
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7

In [ ]:
# OPTIONAL: Run this code cell if you are using Google Colab
# Mount a Google Drive folder so that the data files can be accessed
from google.colab import drive
from google.colab import files
import sys
drive.mount('/content/drive', force_remount=True)
# Change Path
%cd drive/MyDrive/
sys.path.insert(0,'/content/drive/MyDrive//')

Mounted at /content/drive
/content/drive/MyDrive/spoofed_lidar


In [ ]:
# Import libraries and utility functions
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from utils import draw_geometries

In [ ]:
# Load point cloud from .txt file to a NumPy array
initial_pcd = np.loadtxt('cloud.txt').astype(np.float32)
print("Point cloud dimensions are: ", initial_pcd.shape)
np.set_printoptions(precision=3, suppress=True)

Point cloud dimensions are:  (123683, 3)


In [ ]:
# Visualize the point cloud using a 3D web viewer
pcd_object = o3d.geometry.PointCloud()
pcd_object.points = o3d.utility.Vector3dVector(initial_pcd[:, 0:3])
draw_geometries([pcd_object], show_axes=True)

Output hidden; open in https://colab.research.google.com to view.

# 2. Point Cloud Filtering

In [ ]:
# Eliminate Ground Points

def filter_ground(cloud, ground_level=0):
    return cloud[cloud[:, 2] > ground_level, :]


# Filter out ground points
pcd_filtered = filter_ground(initial_pcd, -1.2)
print("Filtered point cloud from %d points to %d points" % (len(initial_pcd), len(pcd_filtered)))

# Visualize point cloud after filtering

pcd_object.points = o3d.utility.Vector3dVector(pcd_filtered[:, 0:3])
draw_geometries([pcd_object], show_axes=True)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Eliminate Distant Points

def filter_by_distance(cloud, distance=10):
    mask = np.sum(cloud[:,:2]**2, axis=1) < distance * distance
    #mask = np.sqrt(np.sum(cloud[:, :2]**2, axis=1)) < distance
    return cloud[mask, :]


pcd_filtered_2 = filter_by_distance(pcd_filtered, 10)
print("Filtered point cloud from %d points to %d points" % (len(pcd_filtered), len(pcd_filtered_2)))

pcd_object.points = o3d.utility.Vector3dVector(pcd_filtered_2)
draw_geometries([pcd_object], show_axes=True)

Filtered point cloud from 40434 points to 9049 points


In [ ]:
# @title
# TODO: implementation function to perform Euclidean clustering at a specified threshold in meters
def euclidean_clustering(cloud, threshold=0.5):
    cluster_labels = np.zeros(len(cloud), dtype=int)
    cluster_idx = 1
    for i in range(len(cloud)):
        if cluster_labels[i] > 0:
            continue
        Q = [i]
        cluster_labels[i] = cluster_idx
        while len(Q) > 0:
            distances = np.sum((cloud - cloud[Q[-1]])**2, axis=1)
            neighbor_mask = distances < threshold * threshold
            neighbor_mask = np.logical_and(neighbor_mask, cluster_labels==0)
            cluster_labels[neighbor_mask] = cluster_idx
            del Q[-1]
            Q.extend(np.nonzero(neighbor_mask)[0])
        cluster_idx += 1
    return cluster_labels


In [ ]:
# Perform Clustering on Point Cloud

cluster_labels = euclidean_clustering(pcd_filtered_2)

print('Found %d clusters from %d points'%(cluster_labels.max(), len(pcd_filtered_2)))
pcd_object.points = o3d.utility.Vector3dVector(pcd_filtered_2)
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(cluster_labels)+1,3))
pcd_object.colors = o3d.utility.Vector3dVector(obj_color[cluster_labels])
draw_geometries([pcd_object])

Found 16 clusters from 9049 points


# 4. Pointcloud Data Injection

In [ ]:
# TODO: Implement Injection of False Data (Spoofed MotorBike and Human)


# Load in the the mesh "bike.ply" using the o3d.io.read_triangle_mesh() method and store in a variable named "mesh"
mesh = o3d.io.read_triangle_mesh("bike.ply")

# Use the "sample_points_uniformly()" method of the mesh object to set the density of the mesh. Use the 'number_of_points' parameter
# and set it to 5000. Store the result in a variable named "bike".
bike = mesh.sample_points_uniformly(number_of_points=5000)

# Represent the mesh as a numpy array and store it in a variable named "bike_pcd".
bike_pcd = np.asarray(bike.points)

# Uncomment the next two lines of code to pre-process the pointcloud.
bike_pcd = bike_pcd * 0.001
rotation_matrix_x = np.array([[1, 0, 0],[0, np.cos(np.deg2rad(90)), -np.sin(np.deg2rad(90))],[0, np.sin(np.deg2rad(90)), np.cos(np.deg2rad(90))]])
bike_pcd[:, 1] -= 6
bike_pcd = np.dot(bike_pcd, rotation_matrix_x.T)
bike_pcd[:, 2] += 4.3


# Load in the the mesh "full_body.ply" using the o3d.io.read_triangle_mesh() method and store in a variable named "person_mesh"
person_mesh = o3d.io.read_triangle_mesh("full_body.ply")

# Use the "sample_points_uniformly()" method of the mesh object to set the density of the mesh. Use the 'number_of_points' parameter
# and set it to 2000. Store the result in a variable named "person".
person = person_mesh.sample_points_uniformly(number_of_points=2000)

# Represent the mesh as a numpy array and store it in a variable named "person_pcd".
person_pcd = np.asarray(person.points)

# Uncomment the next five lines of code to pre-process the pointcloud.
person_pcd = person_pcd * 0.001
person_pcd[:, 0] -= 2
rotation_matrix_x = np.array([[1, 0, 0],[0, np.cos(np.deg2rad(90)), -np.sin(np.deg2rad(90))],[0, np.sin(np.deg2rad(90)), np.cos(np.deg2rad(90))]])
person_pcd = np.dot(person_pcd, rotation_matrix_x.T)
person_pcd = person_pcd[person_pcd[:, 2] > -0.7,:]


In [ ]:
# Use the np.vstack() function to create a union of the three pointcloud objects: pcd_filtered_2, bike_pcd, and person_pcd.
# Store the result in a variable named 'spoofed_pointcloud_data'.
spoofed_pointcloud_data = np.vstack((pcd_filtered_2, bike_pcd, person_pcd))

# Perform clustering on the new spoofed pointcloud using the euclidean_clustering() function you created in Lab 2.
# Store the result in a variable named 'spoofed_labels'.
spoofed_labels = euclidean_clustering(spoofed_pointcloud_data)

# Print out the shape of both the original pointcloud object and the new spoofed pointcloud object to see how they contrast.
print(pcd_filtered_2.shape)
print(spoofed_pointcloud_data.shape)

(9049, 3)
(15551, 3)


In [ ]:
# Visualize the spoofed pointcloud
spoofed_pcd = o3d.geometry.PointCloud()
spoofed_pcd.points = o3d.utility.Vector3dVector(spoofed_pointcloud_data)
print('Found %d clusters from %d points'%(spoofed_labels.max(), len(spoofed_pointcloud_data)))
spoofed_labels = euclidean_clustering(spoofed_pointcloud_data)
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(spoofed_labels)+1,3))
spoofed_pcd.colors = o3d.utility.Vector3dVector(obj_color[spoofed_labels])
draw_geometries([spoofed_pcd])

Found 18 clusters from 15551 points


## 5. Simulating a Laser Spoofer

In [ ]:
# TODO: Simulate a Laser Spoofer

# Laser Position
laser_pos = np.array([5.0, 5.0, 0.0])

# Implement the 'generate_spoof_laser_points()' function. This function will create spoofed laser points based on
# the position of the laser, number of points, and a spread radius.
def generate_spoof_laser_points(laser_pos, num_points=40, spread_radius=0.4):
    # x_vals and y_vals already completed
    x_vals = np.random.uniform(laser_pos[0] - spread_radius, laser_pos[0] + spread_radius/10, num_points)
    y_vals = np.random.uniform(laser_pos[1] - spread_radius, laser_pos[1] + spread_radius/10, num_points)

    # Use the np.random.uniform() function to create a distribution between laser_pos[2] - spread_radius*5
    # and laser_pos[2] + spread_radius*5. Store the result in a variable named z_vals.
    z_vals = np.random.uniform(laser_pos[2] - spread_radius*5, laser_pos[2] + spread_radius*5, num_points)

    # Use the np.vstack() function to create a union of all three axis values. Store in a variable named spoofed_points.
    spoofed_points = np.vstack((x_vals, y_vals, z_vals))

    # Set spoofed_points equal to it's transpose (spoofed_points.T).
    spoofed_points = spoofed_points.T

    # Return the new spoofed points.
    return spoofed_points

In [ ]:
# Generate a set of spoofed laser points. Pass in laser_pos as an argument and store the result in a variable named
# spoofed_laser_points.
spoofed_laser_points = generate_spoof_laser_points(laser_pos)

# Use the np.vstack() function to create a union of pcd_filtered_2 and spoofed_laser_points.
# Store the result in a variable named combined_points.
combined_points = np.vstack((pcd_filtered_2, spoofed_laser_points))

# Print out the shape of pcd_filtered_2 and combined_points
print(pcd_filtered_2.shape)
print(combined_points.shape)



(9049, 3)
(9089, 3)


In [ ]:
# Visualize the spoofed laser points
spoofed_pcd = o3d.geometry.PointCloud()
spoofed_pcd.points = o3d.utility.Vector3dVector(combined_points)
draw_geometries([spoofed_pcd])

In [ ]:
# TODO: Perform clustering on combined_points with a threshold of 0.7.
combined_labels = euclidean_clustering(combined_points, threshold=0.7)

# Visualize with Clustering
print('Found %d clusters from %d points'%(combined_labels.max(), len(combined_points)))
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(combined_labels)+1,3))
spoofed_pcd.colors = o3d.utility.Vector3dVector(obj_color[combined_labels])
draw_geometries([spoofed_pcd])

Found 16 clusters from 9089 points


## 5. Defense against Laser Spoofer

In [ ]:
# TODO: Implement functions for spoofing defense


# TODO: Implement the get_volume function. This function returns the volume of a passed in np array of 3D points, arr.
# Formula for Volume: L * W * H
def get_volume(arr):
  min_x = np.min(arr[:,0])
  max_x = np.max(arr[:,0])
  min_y = np.min(arr[:,1])
  max_y = np.max(arr[:,1])
  min_z = np.min(arr[:,2])
  max_z = np.max(arr[:,2])

  return (max_x - min_x) * (max_y - min_y) * (max_z - min_z)

# TODO: Implement the filter_spoofed_points() function.
def filter_spoofed_points(cloud, labels, threshold):
  new_cloud = None
  new_labels = None
  for i in range(labels.max()):
    cluster_cloud = cloud[labels == (i+1)]
    cluster_volume = get_volume(cluster_cloud)
    cluster_points = len(cluster_cloud)
    cluster_density = cluster_points / cluster_volume
    if cluster_density < threshold:
      new_cloud = np.delete(cloud, labels == (i+1), axis=0)
      new_labels = np.delete(labels, labels == (i+1), axis=0)
  return new_cloud, new_labels


In [ ]:
# TODO: Run the new filter_spoofed_points() function and store the results in variables filtered_cloud and filtered_labels, respectively.
#       Pass in a threshold of 70 as an argument.
filtered_cloud, filtered_labels = filter_spoofed_points(combined_points, combined_labels, threshold=70)

In [ ]:
# Visualize the filtered pointcloud data with no spoofed points
print('Found %d clusters from %d points'%(filtered_labels.max(), len(filtered_cloud)))
filtered_pcd = o3d.geometry.PointCloud()
filtered_pcd.points = o3d.utility.Vector3dVector(filtered_cloud)
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(filtered_labels)+1,3))
filtered_pcd.colors = o3d.utility.Vector3dVector(obj_color[filtered_labels])
draw_geometries([filtered_pcd])

Found 15 clusters from 9049 points
